# v6b ReDo fix (GPU 1): continual from phase 60, resetting dead neurons
The continual run from phase 60 (just before it breaks down), same recipe, with one change: at the start of every phase and every 5,000 steps, MLP neurons that fire on fewer than 1% of tokens (checked on the English source and on Hindi, Chinese AND Arabic text, so a neuron used by only one language is not touched) get fresh incoming weights and zero outgoing weights (ReDo, Sokar et al. 2023).
Question: does resetting dead neurons stop the breakdown after phase 60, and does the self survive?
**Just Run All.** Rerunning resumes from the last saved phase. Checkpoints: `runs/v6b_cont_redo_from60/ckpt/`.

In [ ]:
import os, sys, glob, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "1"     # run this cell FIRST in a fresh kernel
os.environ["HF_HUB_DISABLE_XET"] = "1"          # HF uploads failed overnight in the new "xet" path; use the classic upload
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common_v6 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
RUN = "v6b_cont_redo_from60"

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):                          # retry: HF sometimes drops the connection
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)

SRC_RUN, SRC_PHASE = "v4b_continual_s0", 60
if not glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt"):
    name = next(f for f in api.list_repo_files(HF_REPO) if f.startswith(f"{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_"))
    hf_hub_download(HF_REPO, name, local_dir="runs")
INIT = glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt")[0]
print("branch from:", INIT)

# FLORES dev + devtest
from datasets import load_dataset
for l, code in {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}.items():
    for split in ["dev", "devtest"]:
        p = f"data/flores/{split}.{l}.jsonl"
        if not os.path.exists(p):
            os.makedirs("data/flores", exist_ok=True)
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for t in ds["text"]:
                    f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
missing = [p for p in [f"data/en-{l}/{s}.jsonl" for l in LANGS for s in ["train", "val"]] if not os.path.exists(p)]
print("data OK" if not missing else f"MISSING: {missing}")

In [ ]:
C = RunConfig(run_name=RUN, mode="continual", order=("hi", "zh", "ar"), init_ckpt=INIT, start_phase=SRC_PHASE,
              n_phases=90, seed=0, time_cap_h=120.0, hf_repo=HF_REPO, redo_every=5000, redo_thresh=0.01)
print(C)

In [ ]:
D = load_data(train_langs=["hi", "zh", "ar"])

In [ ]:
model = train_run(C, D)

In [ ]:
print_run_summary(RUN)